# FastText

FastText에서는 각 단어는 글자 단위 n-gram의 구성으로 취급한다.

예를 들어서 n을 3으로 잡은 트라이그램(tri-gram)의 경우, 

apple은 app, ppl, ple로 분리하고 이들을 벡터로 만든다. 

더 정확히는 시작과 끝을 의미하는 <, >를 도입하여 

아래의 5개 내부 단어(subword) 토큰을 벡터로 만든다.

`<ap`, `app`, `ppl`, `ple`, `le>` 

### 장점

#### 1. 모르는 단어에 대한 대응이 가능하다.

가령 예를 들어서 birthplace(출생지)라는 단어가 학습되지 않은 상태라고 할 때

Word2Vec, GloVe는 모르는 단어에 대해 제대로 대응할 수 없지만

FastText는 birth와 place라는 내부 단어로부터 단어 벡터를 추출할 수 있다.

#### 2. 단어 집합 내 빈도 수가 적었던 단어(Rare Word)에 대한 대응이 가능하다.



In [1]:
import re
from lxml import etree
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from gensim.models import FastText

# XML 파일 열기
target_xml = open("ted_en-20160408.xml", "r", encoding="UTF8")
target_text = etree.parse(target_xml)

# <content> 태그 내용 추출
parse_text = "\n".join(target_text.xpath("//content/text()"))

# 괄호 안 문자열 제거
content_text = re.sub(r"\([^)]*\)", "", parse_text)

# 문장 토큰화
sent_text = sent_tokenize(content_text)

# 소문자 변환 + 특수문자 제거
normalized_text = []

for sentence in sent_text:
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z0-9]+", " ", sentence)
    normalized_text.append(sentence)

# 단어 토큰화
result = [word_tokenize(sentence) for sentence in normalized_text]

print("총 문장 수 :", len(result))

print("\n첫 번째 문장:")
print(result[0])

print("\n첫 3개 문장:")
for line in result[:3]:
    print(line)

# =========================
# FastText 학습
# =========================

model = FastText(
    result,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

# =========================
# 유사 단어 확인
# =========================

print("\n[man과 유사한 단어]")
print(model.wv.most_similar("man"))

# =========================
# OOV 테스트
# =========================

print("\n[electrofishing과 유사한 단어]")
print(model.wv.most_similar("electrofishing"))

# vocab 포함 여부 확인
print("\n'electrofishing' vocab 포함 여부:")
print("electrofishing" in model.wv.key_to_index)

# =========================
# 모델 저장
# =========================

model.save("fasttext_ted.model")

# =========================
# 모델 다시 로드
# =========================

loaded_model = FastText.load("fasttext_ted.model")

print("\n[로드한 모델 테스트]")
print(loaded_model.wv.most_similar("electrofishing"))

총 문장 수 : 273424

첫 번째 문장:
['here', 'are', 'two', 'reasons', 'companies', 'fail', 'they', 'only', 'do', 'more', 'of', 'the', 'same', 'or', 'they', 'only', 'do', 'what', 's', 'new']

첫 3개 문장:
['here', 'are', 'two', 'reasons', 'companies', 'fail', 'they', 'only', 'do', 'more', 'of', 'the', 'same', 'or', 'they', 'only', 'do', 'what', 's', 'new']
['to', 'me', 'the', 'real', 'real', 'solution', 'to', 'quality', 'growth', 'is', 'figuring', 'out', 'the', 'balance', 'between', 'two', 'activities', 'exploration', 'and', 'exploitation']
['both', 'are', 'necessary', 'but', 'it', 'can', 'be', 'too', 'much', 'of', 'a', 'good', 'thing']

[man과 유사한 단어]
[('batman', 0.8234599232673645), ('woman', 0.8014000654220581), ('ekman', 0.7775717377662659), ('kahneman', 0.767944872379303), ('foreman', 0.7567362189292908), ('shaman', 0.7566971182823181), ('salman', 0.7534598708152771), ('gottman', 0.7513454556465149), ('hoffman', 0.7496535778045654), ('lehman', 0.747101366519928)]

[electrofishing과 유사한 단어]
[('elec